# MG2 on a free T4 (Kaggle or Colab)

Follow `docs/checkpoints/mg2/CLOUD_RUNBOOK.md`. Set the two values in the next cell, then run every
cell in order (on Kaggle: **Save Version → Save & Run All**, so it runs with the browser closed).
The evaluator answers (`benchmarks/hidden`) are never uploaded; results are scored on the laptop.

In [ ]:
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

# The only two values to set. QUEUE is one of the files in docs/checkpoints/mg2/queue/t4/.
QUEUE = "docs/checkpoints/mg2/queue/t4/queue-t4-rules-on-rest.json"
HOURS = 10.5  # stop starting trials after this long, so the save finishes inside the 12 h limit

STARTED = time.time()
BRANCH = "codex/mg2-cloud-t4"
REPO = Path("/opt/ReCycleGuard")
KAGGLE = Path("/kaggle").is_dir()
if KAGGLE:
    PERSIST = Path("/kaggle/working")
    # A previous run's output, attached with "Add Input", appears under /kaggle/input.
    previous = sorted(Path("/kaggle/input").glob("*/work"))
    RESTORE = previous[0].parent if previous else None
else:
    from google.colab import drive

    drive.mount("/content/drive")
    PERSIST = RESTORE = Path("/content/drive/MyDrive/recycleguard-work")
print("platform:", "kaggle" if KAGGLE else "colab", "| persist:", PERSIST, "| restore:", RESTORE)


def sh(*command, **options):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], check=True, **options)

In [ ]:
# 1. Code: the pushed branch, at one recorded revision.
if not REPO.is_dir():
    sh("git", "clone", "--branch", BRANCH, "https://github.com/R-V-Abhishek/ReCycleGuard.git", REPO)
sh("git", "-C", REPO, "rev-parse", "HEAD")
sh("git", "-C", REPO, "status", "--short")

In [ ]:
# 2. Earlier records back into /work (nothing happens on the first run).
if RESTORE is not None:
    sh("sh", REPO / "scripts/cloud/persist.sh", "restore", RESTORE)

In [ ]:
# 3. Bootstrap: GPU, Python, llama-server, weights, model server, probe, queue check (20-30 min).
sh("sh", REPO / "scripts/cloud/bootstrap.sh", QUEUE, cwd=REPO)

In [ ]:
# 4. Copy /work to the persistent folder every 10 minutes, then start the queue.
persist = subprocess.Popen(
    ["sh", str(REPO / "scripts/cloud/persist.sh"), "every", str(PERSIST)],
    stdout=open("/work/_model/persist.log", "ab"),
    stderr=subprocess.STDOUT,
)
name = json.loads((REPO / QUEUE).read_text())["name"]
uv = shutil.which("uv") or os.path.expanduser("~/.local/bin/uv")
queue = subprocess.Popen(
    [uv, "run", "scripts/run_queue.py", "--host", QUEUE],
    cwd=REPO,
    stdout=open(f"/work/{name}.queue.log", "ab"),
    stderr=subprocess.STDOUT,
)
print("queue", name, "started")

In [ ]:
# 5. Wait. After HOURS, KILL stops every session before its next trial; nothing finished is lost.
folder = Path("/work") / name
killed = False
while queue.poll() is None:
    if not killed and time.time() - STARTED > HOURS * 3600:
        (folder / "KILL").touch()
        killed = True
        print("time limit: KILL written, waiting for running trials to end", flush=True)
    status = Path("/work/_status/status.md")
    if status.is_file():
        print(status.read_text().split("\n\n")[1], flush=True)
    time.sleep(300)
print("queue exit code", queue.returncode)
print(Path(f"/work/{name}.queue.log").read_text()[-2000:])

In [ ]:
# 6. Final copy. KILL is removed first, so the next session resumes instead of stopping.
(folder / "KILL").unlink(missing_ok=True)
persist.terminate()
sh("sh", REPO / "scripts/cloud/persist.sh", "save", PERSIST)
print((Path("/work/_status/status.md")).read_text())